In [1]:
import qlib
import pandas as pd
import numpy as np
from qlib.constant import REG_CN
from qlib.utils import exists_qlib_data, init_instance_by_config
from qlib.workflow import R
from qlib.workflow.record_temp import SignalRecord, PortAnaRecord
from qlib.utils import flatten_dict
from qlib.contrib.model.gbdt import LGBModel
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.contrib.data.handler import Alpha158
from qlib.data import D

ModuleNotFoundError.  PyTorch models are skipped (optional: maybe installing pytorch can fix it).


In [2]:
qlib.init(provider_uri="~/.qlib/qlib_data/cn_data", region=REG_CN)

[80968:MainThread](2025-08-12 21:54:44,003) INFO - qlib.Initialization - [config.py:451] - default_conf: client.
[80968:MainThread](2025-08-12 21:54:44,015) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.
[80968:MainThread](2025-08-12 21:54:44,016) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/dynam1te/.qlib/qlib_data/cn_data')}


In [3]:
# 价格 / factor 对应与前复权的结果

df = D.features(instruments=['SH601033'], fields=['$open', '$close', "$factor"], start_time='2025-08-01', end_time='2025-08-06')

print(df)

                          $open    $close   $factor
instrument datetime                                
SH601033   2025-08-01  0.734431  0.739757  0.048413
           2025-08-04  0.736512  0.744259  0.048423
           2025-08-05  0.744608  0.746060  0.048414
           2025-08-06  0.747997  0.746060  0.048414


In [97]:
# 训练

market = "csi300"
benchmark = "SH000300"

model = LGBModel(
    loss="mse",
    colsample_bytree=0.8879,
    learning_rate=0.0421,
    subsample=0.8789,
    lambda_l1=205.6999,
    lambda_l2=580.9768,
    max_depth=8,
    num_leaves=210,
    num_threads=20,
    keep_training_booster=True
)

dataset = DatasetH( 
    handler=Alpha158(
        instruments="csi300",
        start_time="2018-01-01",
        end_time="2025-08-05",
        fit_start_time="2023-01-01",
        fit_end_time="2023-12-31",
        label = ["Ref($close, -1)/$close - 1"]
        #label = ["$close/$close - 1"]
    ),
    segments={
	    "train": ("2018-01-01", "2023-12-31"),
	    "valid": ("2024-01-01", "2024-12-31"),
        "test": ("2025-01-01", "2025-08-06"),
    },
)

# start exp to train model
with R.start(experiment_name="train_model"):
    #R.log_params(**flatten_dict(task))
    model.fit(dataset)
    R.save_objects(trained_model=model)
    rid = R.get_recorder().id

[80968:MainThread](2025-08-14 20:45:44,996) INFO - qlib.timer - [log.py:127] - Time cost: 35.309s | Loading data Done
[80968:MainThread](2025-08-14 20:45:45,813) INFO - qlib.timer - [log.py:127] - Time cost: 0.205s | DropnaLabel Done
[80968:MainThread](2025-08-14 20:45:46,650) INFO - qlib.timer - [log.py:127] - Time cost: 0.834s | CSZScoreNorm Done
[80968:MainThread](2025-08-14 20:45:46,654) INFO - qlib.timer - [log.py:127] - Time cost: 1.656s | fit & process data Done
[80968:MainThread](2025-08-14 20:45:46,655) INFO - qlib.timer - [log.py:127] - Time cost: 36.969s | Init data Done
[80968:MainThread](2025-08-14 20:45:46,668) INFO - qlib.workflow - [exp.py:258] - Experiment 546341030589329388 starts running ...
[80968:MainThread](2025-08-14 20:45:46,681) INFO - qlib.workflow - [recorder.py:345] - Recorder 14356569f5794f0abb935fbbafd1ad19 starts running under Experiment 546341030589329388 ...


Training until validation scores don't improve for 50 rounds
[20]	train's l2: 0.990259	valid's l2: 0.993887
[40]	train's l2: 0.986563	valid's l2: 0.992759
[60]	train's l2: 0.983858	valid's l2: 0.992165
[80]	train's l2: 0.981482	valid's l2: 0.991759
[100]	train's l2: 0.979349	valid's l2: 0.991457
[120]	train's l2: 0.977465	valid's l2: 0.991244
[140]	train's l2: 0.975583	valid's l2: 0.99123
[160]	train's l2: 0.973829	valid's l2: 0.991151
[180]	train's l2: 0.972122	valid's l2: 0.99116
[200]	train's l2: 0.970437	valid's l2: 0.991252
Early stopping, best iteration is:
[158]	train's l2: 0.973995	valid's l2: 0.991132


[80968:MainThread](2025-08-14 20:46:06,398) INFO - qlib.timer - [log.py:127] - Time cost: 0.431s | waiting `async_log` Done


In [98]:
df = dataset.prepare("train", col_set=["feature", "label"], data_key=DataHandlerLP.DK_L)

x, y = df["feature"], df["label"]

In [99]:
df

feature                                          \
                           KMID      KLEN     KMID2       KUP      KUP2   
datetime   instrument                                                     
2018-01-02 SH600000    0.008723  0.013481  0.647058  0.003965  0.294121   
           SH600008    0.009709  0.017476  0.555557  0.003883  0.222221   
           SH600009   -0.013333  0.026667 -0.500002  0.007333  0.275001   
           SH600010    0.016260  0.024390  0.666668  0.004065  0.166666   
           SH600011    0.014563  0.022654  0.642858  0.004854  0.214285   
...                         ...       ...       ...       ...       ...   
2023-12-29 SZ300919    0.010905  0.020165  0.540817  0.004115  0.204082   
           SZ300957    0.001763  0.019251  0.091605  0.011903  0.618322   
           SZ300979    0.013087  0.026174  0.499998  0.010778  0.411766   
           SZ300999   -0.000599  0.011078 -0.054051  0.004790  0.432432   
           SZ301269    0.009152  0.021928  0.417388  0.006197  0.282609   

                                                                         ...  \
                           KLOW     KLOW2      KSFT     KSFT2     OPEN0  ...   
datetime   instrument                                                    ...   
2018-01-02 SH600000    0.000793  0.058820  0.005551  0.411757  0.991352  ...   
           SH600008    0.003883  0.222221  0.009709  0.555557  0.990385  ...   
           SH600009    0.006000  0.224997 -0.014667 -0.550007  1.013514  ...   
           SH600010    0.004065  0.166666  0.016260  0.666668  0.984000  ...   
           SH600011    0.003236  0.142858  0.012945  0.571430  0.985646  ...   
...                         ...       ...       ...       ...       ...  ...   
2023-12-29 SZ300919    0.005144  0.255101  0.011934  0.591837  0.989212  ...   
           SZ300957    0.005584  0.290073 -0.004556 -0.236645  0.998240  ...   
           SZ300979    0.002309  0.088236  0.004619  0.176467  0.987082  ...   
           SZ300999    0.005689  0.513517  0.000299  0.027034  1.000599  ...   
           SZ301269    0.006578  0.300002  0.009534  0.434781  0.990931  ...   

                                                                         \
                        VSUMN10   VSUMN20   VSUMN30   VSUMN60    VSUMD5   
datetime   instrument                                                     
2018-01-02 SH600000    0.388162  0.508938  0.531578  0.509717  0.226996   
           SH600008    0.431804  0.484225  0.486605  0.507666  0.026581   
           SH600009    0.428595  0.468053  0.550978  0.510013  0.368400   
           SH600010    0.456158  0.531998  0.488363  0.494932 -0.113531   
           SH600011    0.510928  0.554349  0.498618  0.490071 -0.428183   
...                         ...       ...       ...       ...       ...   
2023-12-29 SZ300919    0.440239  0.476402  0.492429  0.488660 -0.039531   
           SZ300957    0.529650  0.521841  0.512245  0.490720  0.021413   
           SZ300979    0.514340  0.524758  0.530498  0.494032 -0.709749   
           SZ300999    0.460011  0.616508  0.482271  0.499917  0.081677   
           SZ301269    0.478421  0.488480  0.496295  0.502985 -0.070268   

                                                               \
                        VSUMD10   VSUMD20   VSUMD30   VSUMD60   
datetime   instrument                                           
2018-01-02 SH600000    0.223676 -0.017876 -0.063155 -0.019434   
           SH600008    0.136393  0.031550  0.026790 -0.015332   
           SH600009    0.142809  0.063894 -0.101957 -0.020025   
           SH600010    0.087683 -0.063996  0.023274  0.010136   
           SH600011   -0.021856 -0.108698  0.002763  0.019859   
...                         ...       ...       ...       ...   
2023-12-29 SZ300919    0.119521  0.047196  0.015142  0.022680   
           SZ300957   -0.059299 -0.043682 -0.024490  0.018560   
           SZ300979   -0.028681 -0.049517 -0.060995  0.011936   
           SZ300999  